# WA1 Water Footprint Model -- Manager Notebook

Orchestrates training of the Cross-Attention Geo-Aware Water Footprint Network.
All model logic lives in `src/` -- this notebook only imports and calls.

## 1. Initialization

In [ ]:
# -- Configuration --
GITHUB_USERNAME = ""  # Fill in
GITHUB_TOKEN = ""     # Fill in (fine-grained PAT)
REPO_URL = ""         # Fill in

import os
import subprocess

IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    DRIVE_BASE = "/content/drive/MyDrive/ESPResso-V2"
    os.makedirs(DRIVE_BASE, exist_ok=True)
    print(f"Drive mounted at {DRIVE_BASE}")
else:
    DRIVE_BASE = None
    print("Running locally")

# -- Preflight validation --
errors = []
if IN_COLAB:
    if not GITHUB_USERNAME:
        errors.append("GITHUB_USERNAME is empty -- fill in cell above")
    if not GITHUB_TOKEN:
        errors.append("GITHUB_TOKEN is empty -- fill in cell above")
    if not REPO_URL:
        errors.append("REPO_URL is empty -- fill in cell above")

for pkg in ["torch", "pandas", "numpy", "sklearn"]:
    try:
        __import__(pkg)
    except ImportError:
        errors.append(f"Required package '{pkg}' not importable")

if IN_COLAB:
    lfs_check = subprocess.run(["git", "lfs", "version"], capture_output=True)
    if lfs_check.returncode != 0:
        errors.append("git-lfs not installed. Run: !apt-get install git-lfs")

assert not errors, "Preflight failed:\n  " + "\n  ".join(errors)
print("Preflight passed")

## 2. Clone / Update Repository

In [ ]:
if IN_COLAB:
    REPO_DIR = "/content/ESPResso-V2"
    if not os.path.exists(REPO_DIR):
        auth_url = REPO_URL.replace("https://", f"https://{GITHUB_USERNAME}:{GITHUB_TOKEN}@")
        subprocess.run(
            ["git", "clone", auth_url, REPO_DIR],
            check=True,
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
        )
    else:
        subprocess.run(["git", "pull"], check=True, cwd=REPO_DIR)
    os.chdir(REPO_DIR)
else:
    REPO_DIR = os.getcwd()

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Working directory: {os.getcwd()}")

## 3. Check Data Cache

In [ ]:
def _is_lfs_pointer(path):
    """True if file is a Git LFS pointer stub, not actual data."""
    if not os.path.exists(path):
        return False
    if os.path.getsize(path) > 1024:
        return False
    with open(path, "r") as f:
        return f.readline().startswith("version https://git-lfs")

DATA_PATH = os.path.join(REPO_DIR, "model", "data", "water_footprint.csv")
data_ready = os.path.exists(DATA_PATH) and not _is_lfs_pointer(DATA_PATH)

if IN_COLAB:
    CACHE_PATH = os.path.join(DRIVE_BASE, "data", "water_footprint.csv")
    if not data_ready and os.path.exists(CACHE_PATH) and not _is_lfs_pointer(CACHE_PATH):
        os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
        !cp {CACHE_PATH} {DATA_PATH}
        data_ready = True
        print(f"Restored data from Drive cache")
    if not data_ready:
        print("Pulling data via Git LFS...")
        !cd {REPO_DIR} && git lfs pull --include="model/data/**"
        data_ready = os.path.exists(DATA_PATH) and not _is_lfs_pointer(DATA_PATH)
        if data_ready:
            os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
            !cp {DATA_PATH} {CACHE_PATH}
            print("Cached data to Drive")

assert data_ready, f"Data not found or still LFS pointer at {DATA_PATH}"
print(f"Data ready: {DATA_PATH}")

## 4. Smoke Test

In [ ]:
import copy
from model.water_footprint.src.utils.config import WA1Config, set_seed
from model.water_footprint.src.training.checkpoint import smoke_test

config = WA1Config()
config.data_dir = os.path.join(REPO_DIR, "model", "data")

if IN_COLAB:
    config.checkpoint_dir = os.path.join(DRIVE_BASE, "checkpoints", "water_footprint")

# Pass a copy -- smoke_test mutates config (batch_size, max_epochs, checkpoint_dir)
result = smoke_test(copy.deepcopy(config))
assert result["passed"], f"Smoke test failed: {result}"
print(f"Smoke test passed -- train_loss={result['train_loss']:.4f}, gate_A={result['gate_a']:.4f}, gate_F={result['gate_f']:.4f}")

## 5. Check for Existing Checkpoint

In [ ]:
import glob

checkpoint_pattern = os.path.join(config.checkpoint_dir, "checkpoint_*.pt")
existing = sorted(glob.glob(checkpoint_pattern))

if existing:
    print(f"Found {len(existing)} existing checkpoint(s):")
    for cp in existing:
        print(f"  {cp}")
    RESUME_FROM = existing[-1]
    print(f"Will resume from: {RESUME_FROM}")
else:
    RESUME_FROM = None
    print("No existing checkpoints. Training from scratch.")

## 6. Prepare Data

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from model.water_footprint.src.preprocessing.dataset import WaterFootprintDataset
from model.water_footprint.src.preprocessing.transforms import (
    Log1pZScoreTransform, create_splits,
)

set_seed(config.seed)
device = config.device
print(f"Device: {device}")

# Load dataset and split
full_dataset = WaterFootprintDataset(str(config.data_path))
train_set, val_set, test_set = create_splits(
    full_dataset,
    category_names=full_dataset.category_names,
    seed=config.seed,
    train_ratio=config.split_ratios[0],
    val_ratio=config.split_ratios[1],
    test_ratio=config.split_ratios[2],
)
print(f"Split: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}")

# Fit transform on training targets (3 separate arrays as fit() expects)
train_records = [full_dataset.records[i] for i in train_set.indices]
train_raw = np.array([r["wf_raw"] for r in train_records])
train_proc = np.array([r["wf_processing"] for r in train_records])
train_pkg = np.array([r["wf_packaging"] for r in train_records])

transform = Log1pZScoreTransform()
transform.fit(train_raw, train_proc, train_pkg)
print(f"Transform fitted: {transform.state_dict()}")

# DataLoaders
NUM_WORKERS = 4 if IN_COLAB else 0
train_loader = DataLoader(
    train_set, batch_size=config.batch_size, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    val_set, batch_size=config.batch_size, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)
test_loader = DataLoader(
    test_set, batch_size=config.batch_size, shuffle=False,
)
print(f"Loaders ready: {len(train_loader)} train batches, {len(val_loader)} val batches")

## 7. Train

In [ ]:
from model.water_footprint.src.training.model import WA1Model
from model.water_footprint.src.training.loss import UWSOLoss
from model.water_footprint.src.training.trainer import WA1Trainer

# Build model
model = WA1Model(config).to(device)
loss_fn = UWSOLoss(config.huber_delta).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {n_params:,} parameters")

# Build trainer
trainer = WA1Trainer(
    config=config,
    model=model,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    transform=transform,
    device=device,
)

# Resume if checkpoint exists
if RESUME_FROM:
    resumed_epoch = trainer.load_checkpoint(RESUME_FROM)
    print(f"Resumed from epoch {resumed_epoch}")

# Viability check (first 5 epochs)
viable = trainer.viability_check(n_canary=config.canary_epochs)
assert viable, "Viability check failed -- see diagnostics above"
print("Viability check passed. Starting full training...")

# Full training
best_metrics, history = trainer.train(
    max_epochs=config.max_epochs,
    patience=config.patience,
)
print(f"Training complete. Best val loss: {best_metrics['val_loss']:.4f}")

## 8. Evaluate

In [ ]:
from model.water_footprint.src.evaluation.metrics import compute_metrics, per_tier_evaluation

# Load best checkpoint
best_ckpt = sorted(glob.glob(checkpoint_pattern))[-1]
trainer.load_checkpoint(best_ckpt)
print(f"Loaded best checkpoint: {best_ckpt}")

# Test set evaluation (Tier F -- full data)
print("\n--- Test Set (Tier F) ---")
test_loss, test_metrics = trainer.val_epoch(loader=test_loader, tier="F")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

# Per-tier evaluation matrix
print("\n--- Per-Tier Evaluation (Validation Set) ---")
tier_matrix = per_tier_evaluation(
    model=model,
    dataloader=val_loader,
    transform=transform,
    tiers=["A", "B", "C", "D", "E", "F"],
    device=device,
)
print(tier_matrix.to_string())

## 9. Summary

In [ ]:
print("=" * 60)
print("WA1 Water Footprint Model -- Training Summary")
print("=" * 60)
print(f"Parameters: {n_params:,}")
print(f"Best val loss: {best_metrics['val_loss']:.4f}")
print(f"Device: {device}")
print(f"Checkpoint: {config.checkpoint_dir}")
print(f"Experiment log: {os.path.join(config.checkpoint_dir, config.runs_log)}")